# 02 - Limpieza y transformación

Recorre las 6 fuentes ya extraídas en `data/02_intermediate/` (una por cada `src/extract_*.py`) antes de que `src/build_dataset.py` las combine, para ver qué aporta cada una por separado y cómo se resuelven sus distintas coberturas temporales/geográficas.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

import pandas as pd
import matplotlib.pyplot as plt

from config import INTERMEDIATE_DIR, PRIMARY_DIR, MODEL_OUTPUT_DIR

pd.set_option("display.max_columns", 50)


## Fuente 1: panel regional GEIH (TD/TO/TGP/TS por región-semestre, 2010-2025)

In [ ]:
geih = pd.read_csv(INTERMEDIATE_DIR / "geih_regional_panel.csv")
geih.head()

## Fuente 2: informalidad laboral nacional (GEIH-ANEXOS, solo desde 2021)

In [ ]:
lab = pd.read_csv(INTERMEDIATE_DIR / "informalidad_laboral_nacional.csv")
lab

## Fuente 3: informalidad empresarial nacional (IMIE, anual 2019-2024)

In [ ]:
emp = pd.read_csv(INTERMEDIATE_DIR / "informalidad_empresarial_nacional.csv")
emp

## Fuente 4: informalidad de micronegocios (EMICRON, un solo año: 2024)

In [ ]:
micro = pd.read_csv(INTERMEDIATE_DIR / "informalidad_micronegocios_nacional.csv")
micro

## Fuente 5: dispersión regional de TD entre departamentos (2007-2025)

In [ ]:
disp = pd.read_csv(INTERMEDIATE_DIR / "td_dispersion_regional.csv")
disp.tail(10)

## Fuente 6: informalidad laboral real por región (microdato GEIH, desde 2022)

A diferencia de la fuente 2 (nacional), esta sí varía por región — pero cubre un rango de años más corto.

In [ ]:
lab_reg = pd.read_csv(INTERMEDIATE_DIR / "informalidad_laboral_regional.csv")
lab_reg.head(10)

## Cobertura temporal de cada fuente

Esto explica por qué `build_dataset.py` necesita forward/backward-fill por región para las fuentes que no cubren 2010-2025 completo (ver columnas `*_obs` en el panel final, que marcan si el dato de ese semestre es real o rellenado).

In [ ]:
fuentes = {
    "geih_regional (fuente 1)": (geih["anio"].min(), geih["anio"].max()),
    "informalidad_laboral_nacional (f.2)": (lab["anio"].min(), lab["anio"].max()),
    "informalidad_empresarial (f.3)": (emp["anio"].min(), emp["anio"].max()),
    "informalidad_micronegocios (f.4)": (micro["anio"].min(), micro["anio"].max()),
    "td_dispersion_regional (f.5)": (disp["anio"].min(), disp["anio"].max()),
    "informalidad_laboral_regional (f.6)": (lab_reg["anio"].min(), lab_reg["anio"].max()),
}
pd.DataFrame(fuentes, index=["anio_min", "anio_max"]).T